# 🚀 DBT (Data Build Tool) - Complete Detailed Notes

------------------------------------------------------------------------

## 📌 What is DBT?

DBT (Data Build Tool) is an open-source tool used for transforming data
inside a data warehouse using SQL.

-   Focus: **Transform (T) in ELT**
-   Works on top of:
    -   Snowflake
    -   BigQuery
    -   Redshift
    -   Databricks

------------------------------------------------------------------------

## 🧠 Core Philosophy

DBT brings **software engineering practices to data**:

-   Modular SQL
-   Version control (Git)
-   Testing
-   Documentation
-   Reusability

------------------------------------------------------------------------

## ⚙️ How DBT Works

1.  Raw data loaded into warehouse
2.  DBT models transform data
3.  DAG auto-created using dependencies
4.  Output tables used for analytics

------------------------------------------------------------------------

## 🧩 Key Concepts (Detailed)

### 1. Models

Each SQL file = a model

``` sql
-- models/staging/stg_orders.sql
SELECT
    id,
    amount,
    created_at
FROM raw.orders
```

------------------------------------------------------------------------

### 2. Materializations

``` sql
{{ config(materialized='table') }}
```

Types: - view - table - incremental

------------------------------------------------------------------------

### 3. Incremental Model (Production Use)

``` sql
{{ config(materialized='incremental') }}

SELECT *
FROM raw.orders

{% if is_incremental() %}
WHERE updated_at > (SELECT MAX(updated_at) FROM {{ this }})
{% endif %}
```

------------------------------------------------------------------------

### 4. DAG using ref()

``` sql
SELECT *
FROM {{ ref('stg_orders') }}
```

👉 Automatically creates dependency graph

------------------------------------------------------------------------

### 5. Tests

``` yaml
models:
  - name: stg_orders
    columns:
      - name: id
        tests:
          - unique
          - not_null
```

------------------------------------------------------------------------

### 6. Macros

``` sql
{% macro tax(amount) %}
    amount * 0.18
{% endmacro %}
```

Usage:

``` sql
SELECT {{ tax('amount') }} FROM table
```

------------------------------------------------------------------------

### 7. Snapshots (SCD Type 2)

``` sql
{% snapshot orders_snapshot %}

{{
    config(
      target_schema='snapshots',
      unique_key='id',
      strategy='timestamp',
      updated_at='updated_at'
    )
}}

SELECT * FROM raw.orders

{% endsnapshot %}
```

------------------------------------------------------------------------

### 8. Documentation

``` bash
dbt docs generate
dbt docs serve
```

------------------------------------------------------------------------

## 🏗️ Project Structure

    models/
      staging/
      intermediate/
      marts/

    macros/
    tests/
    snapshots/

------------------------------------------------------------------------

## 🔥 Why DBT?

### 1. SQL First

-   Easy for analysts
-   No heavy coding

### 2. ELT Approach

-   Uses warehouse power

### 3. Version Control

-   Git-based workflow

### 4. Testing Layer

-   Ensures data quality

### 5. Modularity

-   Reusable SQL models

------------------------------------------------------------------------

## 📈 Why DBT is Trending

-   Rise of Modern Data Stack
-   Analytics Engineering role
-   Faster delivery
-   Lower infrastructure cost
-   Strong ecosystem

------------------------------------------------------------------------

## ⚠️ Limitations

-   No ingestion
-   SQL-heavy
-   Not ideal for ML
-   Depends on warehouse performance

------------------------------------------------------------------------

## 🏗️ Real Production-Level Example

### Staging Layer

``` sql
-- stg_orders.sql
SELECT
    id,
    customer_id,
    amount,
    created_at
FROM raw.orders
```

------------------------------------------------------------------------

### Intermediate Layer

``` sql
-- int_orders.sql
SELECT
    customer_id,
    COUNT(*) as total_orders,
    SUM(amount) as total_spent
FROM {{ ref('stg_orders') }}
GROUP BY customer_id
```

------------------------------------------------------------------------

### Mart Layer

``` sql
-- mart_customer_revenue.sql
SELECT
    customer_id,
    total_orders,
    total_spent,
    CASE
        WHEN total_spent > 10000 THEN 'High Value'
        ELSE 'Low Value'
    END as segment
FROM {{ ref('int_orders') }}
```

------------------------------------------------------------------------

## 🆚 DBT vs Traditional ETL

  Feature    ETL        DBT
  ---------- ---------- -----------
  Language   Python     SQL
  Engine     External   Warehouse
  Testing    Manual     Built-in
  Cost       High       Lower

------------------------------------------------------------------------

## 🎯 Summary

DBT: - Simplifies transformation - Enables analytics engineering -
Improves data quality - Speeds up development

------------------------------------------------------------------------

## 🚀 Next Steps

-   DBT + Airflow integration
-   DBT with Databricks
-   CI/CD with DBT
-   Advanced macros & packages


# 🚀 DBT CLI vs DBT Cloud vs DBT Canvas (Cloud IDE)

------------------------------------------------------------------------

## 🧠 1. DBT CLI (Core)

### 📌 What is it?

-   Open-source DBT (installed locally)
-   Run via terminal/command line

``` bash
dbt run
dbt test
dbt build
```

------------------------------------------------------------------------

### ⚙️ How it Works

-   You write DBT models locally
-   Use Git for version control
-   Run commands manually or via orchestrators (Airflow, etc.)

------------------------------------------------------------------------

### ✅ Pros

-   Free & open source
-   Full control over environment
-   Flexible integration (Airflow, CI/CD tools)
-   Preferred by data engineers

------------------------------------------------------------------------

### ❌ Cons

-   No UI
-   Manual setup required
-   No built-in scheduler
-   No built-in monitoring

------------------------------------------------------------------------

### 🧠 Best Use Case

-   Engineering-heavy teams
-   Custom pipelines
-   When using Airflow/Prefect

------------------------------------------------------------------------

## ☁️ 2. DBT Cloud

### 📌 What is it?

-   Managed SaaS platform by dbt Labs
-   Provides UI + orchestration + scheduling

------------------------------------------------------------------------

### ⚙️ Features

-   Web-based IDE
-   Job scheduling
-   Built-in logging & monitoring
-   Git integration
-   Role-based access control

------------------------------------------------------------------------

### ✅ Pros

-   No infra setup
-   Easy to use (UI-driven)
-   Built-in scheduler
-   Faster onboarding

------------------------------------------------------------------------

### ❌ Cons

-   Paid (can be expensive at scale)
-   Less control than CLI
-   Vendor lock-in risk

------------------------------------------------------------------------

### 🧠 Best Use Case

-   Startups / analytics teams
-   Teams without strong DevOps
-   Quick setup needed

------------------------------------------------------------------------

## 🧑‍💻 3. DBT Canvas (Cloud IDE)

### 📌 What is it?

-   Browser-based development environment inside DBT Cloud

------------------------------------------------------------------------

### ⚙️ Features

-   Write SQL models in browser
-   Auto DAG visualization
-   Run models interactively
-   Inline documentation
-   Version control (Git-backed)

------------------------------------------------------------------------

### ✅ Pros

-   No local setup
-   Great for analysts
-   Visual debugging
-   Easy collaboration

------------------------------------------------------------------------

### ❌ Cons

-   Limited customization
-   Depends on DBT Cloud
-   Not ideal for complex dev workflows

------------------------------------------------------------------------

### 🧠 Best Use Case

-   Analysts writing transformations
-   Quick debugging
-   Exploring lineage

------------------------------------------------------------------------

## ⚖️ Comparison Table

  ------------------------------------------------------------------------
  Feature             DBT CLI         DBT Cloud          DBT Canvas (IDE)
  ------------------- --------------- ------------------ -----------------
  Type                Open-source     SaaS               Web IDE

  UI                  ❌ No           ✅ Yes             ✅ Yes

  Scheduling          ❌ External     ✅ Built-in        ✅ (via Cloud)

  Cost                Free            Paid               Included in Cloud

  Control             High            Medium             Low

  Ease of Use         Medium          High               Very High

  Best For            Engineers       Teams              Analysts
  ------------------------------------------------------------------------

------------------------------------------------------------------------

## 🔥 Real Production Setup

### Option 1: Enterprise (Most Common)

Airflow → DBT CLI → Data Warehouse

------------------------------------------------------------------------

### Option 2: Modern SaaS Setup

DBT Cloud → Warehouse

------------------------------------------------------------------------

### Option 3: Hybrid

Develop in Canvas → Deploy via CLI/Airflow

------------------------------------------------------------------------

## 🎯 When to Choose What?

### 👉 Choose DBT CLI if:

-   You need full control
-   You already use Airflow
-   You want cost optimization

------------------------------------------------------------------------

### 👉 Choose DBT Cloud if:

-   You want quick setup
-   You don't want infra overhead
-   Team includes analysts

------------------------------------------------------------------------

### 👉 Use DBT Canvas if:

-   You want UI-based development
-   Analysts are contributing
-   Quick debugging is needed

------------------------------------------------------------------------

## 🧠 Final Insight

-   DBT CLI = Engine\
-   DBT Cloud = Platform\
-   DBT Canvas = IDE inside Cloud

------------------------------------------------------------------------

## 🚀 Interview Tip

Most companies: - Use DBT CLI + Airflow (production) - Use DBT
Cloud/Canvas (development & collaboration)


# 🚀 DBT Main Components & Architecture (Complete Notes)

------------------------------------------------------------------------

## 🧠 Main Components of DBT

### 1. Models

SQL files that define transformations.

``` sql
SELECT id, amount, created_at FROM raw.orders;
```

------------------------------------------------------------------------

### 2. ref() Function

Creates dependencies and builds DAG.

``` sql
SELECT * FROM {{ ref('stg_orders') }};
```

------------------------------------------------------------------------

### 3. Materializations

Controls how models are built.

``` sql
{{ config(materialized='table') }}
```

Types: - view - table - incremental

------------------------------------------------------------------------

### 4. Incremental Models

``` sql
{{ config(materialized='incremental') }}

SELECT * FROM raw.orders

{% if is_incremental() %}
WHERE updated_at > (SELECT MAX(updated_at) FROM {{ this }})
{% endif %}
```

------------------------------------------------------------------------

### 5. Tests

``` yaml
models:
  - name: orders
    columns:
      - name: id
        tests:
          - unique
          - not_null
```

------------------------------------------------------------------------

### 6. Macros

``` sql
{% macro discount(amount) %}
    amount * 0.1
{% endmacro %}
```

------------------------------------------------------------------------

### 7. Seeds

Static CSV data loaded into warehouse.

------------------------------------------------------------------------

### 8. Snapshots

``` sql
{% snapshot orders_snapshot %}
SELECT * FROM raw.orders
{% endsnapshot %}
```

------------------------------------------------------------------------

### 9. Documentation

``` bash
dbt docs generate
dbt docs serve
```

------------------------------------------------------------------------

## 🏗️ DBT Architecture

### High-Level Flow

Data Sources → Ingestion → Warehouse → DBT → BI Tools

------------------------------------------------------------------------

### Layered Architecture

#### Staging

``` sql
SELECT id, amount FROM raw.orders;
```

#### Intermediate

``` sql
SELECT customer_id, SUM(amount)
FROM {{ ref('stg_orders') }}
GROUP BY customer_id;
```

#### Mart

``` sql
SELECT customer_id, revenue FROM {{ ref('int_orders') }};
```

------------------------------------------------------------------------

## 🔄 Execution Flow

1.  dbt run\
2.  Build DAG\
3.  Execute models\
4.  Materialize tables

------------------------------------------------------------------------

## 🧪 Testing

``` bash
dbt test
```

------------------------------------------------------------------------

## 🔁 Orchestration

Used with: - Airflow - Prefect - DBT Cloud

------------------------------------------------------------------------

## 🎯 Key Insight

-   DBT does not process data
-   It generates SQL
-   Warehouse does computation

------------------------------------------------------------------------

## ⚡ Summary

Components: - Models - Tests - Macros - Seeds - Snapshots

Architecture: - ELT based - Warehouse driven - Layered transformations


# 🚀 How DBT Works Internally (Deep Dive)

------------------------------------------------------------------------

## 🧠 High-Level Idea

DBT does NOT process data itself.\
It compiles SQL, builds a DAG, and sends queries to the data warehouse.

DBT = Compiler + DAG Builder\
Warehouse = Execution Engine

------------------------------------------------------------------------

## ⚙️ Step-by-Step Internal Working

------------------------------------------------------------------------

### 1. Project Parsing

-   Scans project folders (models, macros, tests, snapshots)
-   Reads SQL and YAML files
-   Identifies dependencies using ref() and source()
-   Builds internal DAG

Example:

``` sql
SELECT * FROM {{ ref('model_a') }}
```

------------------------------------------------------------------------

### 2. Jinja Rendering (Macro Expansion)

Input:

``` sql
SELECT * FROM {{ ref('stg_orders') }}
```

Compiled Output:

``` sql
SELECT * FROM analytics.stg_orders
```

------------------------------------------------------------------------

### 3. Compilation Phase

-   Converts DBT models into pure SQL
-   Stores in target/compiled/

Example:

``` sql
CREATE TABLE analytics.orders AS
SELECT * FROM raw.orders;
```

------------------------------------------------------------------------

### 4. DAG Resolution

-   Determines execution order

Example:

stg_orders → int_orders → mart_orders

------------------------------------------------------------------------

### 5. Execution Phase

-   Connects to warehouse (Snowflake, BigQuery, Databricks)
-   Sends SQL queries for execution

Important: DBT does not process data --- warehouse does.

------------------------------------------------------------------------

### 6. Materialization Logic

Based on config:

``` sql
{{ config(materialized='incremental') }}
```

DBT generates: - CREATE TABLE - CREATE VIEW - INSERT INTO (incremental)

------------------------------------------------------------------------

### 7. Testing

``` bash
dbt test
```

Example SQL generated:

``` sql
SELECT COUNT(*) FROM table WHERE id IS NULL;
```

------------------------------------------------------------------------

### 8. Documentation

``` bash
dbt docs generate
```

-   Generates lineage graph
-   Model metadata

------------------------------------------------------------------------

## 📂 Internal Artifacts (target/)

-   compiled/ → compiled SQL
-   run/ → executed SQL
-   manifest.json → DAG metadata
-   run_results.json → execution logs

------------------------------------------------------------------------

## ⚙️ Execution Behavior

-   Multi-threaded execution
-   Parallel execution for independent models

------------------------------------------------------------------------

## 🧠 Key Internal Concepts

### Manifest File

Stores full DAG and metadata.

### Adapter Layer

Converts SQL for specific warehouse.

### Context Variables

-   ref()
-   source()
-   this
-   target

------------------------------------------------------------------------

## 🔁 Full Internal Flow

1.  Parse project\
2.  Build DAG\
3.  Compile SQL\
4.  Resolve dependencies\
5.  Execute on warehouse\
6.  Run tests\
7.  Generate docs

------------------------------------------------------------------------

## 🎯 Key Insight

DBT is NOT a processing engine.\
It is a SQL compiler and DAG builder.

------------------------------------------------------------------------

## ⚡ Summary

DBT internally: - Parses project - Builds DAG - Compiles SQL - Executes
via warehouse - Validates with tests
